In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tqdm
import sys
import os

In [3]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
from pathlib import Path
figures_dir = Path("scratch") / "figures"
figures_dir.mkdir(exist_ok=True, parents=True)

In [5]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value
    print(f"{key}={os.environ.get(key)}")

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

PATH=/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/.venv/bin:/home/bkantz/.vscode-server/cli/servers/Stable-df53daabb18cd157bdb08c7f01c34df936cf12f4/server/bin/remote-cli:/home/bkantz/.bun/bin:/home/bkantz/cmake-4.2.3-linux-x86_64/bin:/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/qlever/build/release:/home/bkantz/bin:/home/bkantz/.local/bin:/usr/local/cuda-13/bin:/home/bkantz/miniconda3/condabin:/home/bkantz/.nvm/versions/node/v23.1.0/bin:/home/bkantz/.cargo/bin:/usr/local/bin:/usr/bin:/bin:/usr/games:/home/bkantz/.local/bin:/home/bkantz/.local/bin$:/home/bkantz/apache-jena/apache-jena-5.2.0/bin:/home/bkantz/apache-jena-fuseki-5.2.0/bin:/home/bkantz/agents/bin:/home/bkantz/llama.cpp/build/bin:/usr/local/texlive/2025/bin/x86_64-linux:/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/.venv/bin:/home/bkantz/.bun/bin:/home/bkantz/cmake-4.2.3-linux-x86_64/bin:/mnt/spinner/spinner/cgv/hereditary/Dense-KG-Vector-DB/qlever/build/release:/usr/local/cuda-13/bin:/home/bkantz/.vscode-server/data/Us

In [6]:
sys.path.append(".")

In [7]:
from utils.datasets import SimpleSet, BerlinSparqlBenchmark
from utils.dbs.qlever import QleverDB
from utils.dbs.fuseki import FusekiDB
from utils.dbs.base_db import BaseDB
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.fuseki_native import FusekiDBNative
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import pandas as pd
import numpy as np
from utils.datasets.base_dataset import DataTensor

2026-08-26 10:11:57,957 - INFO - Loading faiss with AVX512 support.
2026-08-26 10:11:57,958 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-08-26 10:11:57,958 - INFO - Loading faiss with AVX2 support.
2026-08-26 10:11:57,958 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-08-26 10:11:57,959 - INFO - Loading faiss.
2026-08-26 10:11:57,992 - INFO - Successfully loaded faiss.


In [8]:
powers = np.arange(0, 6)  # extend on a more powerful machine
sizes = 10**powers

In [9]:
datasets: dict[int, BerlinSparqlBenchmark] = {}
raw_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Running BDSDM generation for size {size}...")
    dataset = BerlinSparqlBenchmark(base_dir=Path(f"./data/bsbm_{power}"), n=size)
    dataset.setup()
    datasets[power] = dataset
    # raw_sizes[power] = dataset.get_triple_count()

2026-08-26 10:11:58,098 - INFO - BSBM dataset already exists in data/bsbm_0, skipping generation
2026-08-26 10:11:58,099 - INFO - BSBM dataset already exists in data/bsbm_1, skipping generation
2026-08-26 10:11:58,099 - INFO - BSBM dataset already exists in data/bsbm_2, skipping generation
2026-08-26 10:11:58,100 - INFO - BSBM dataset already exists in data/bsbm_3, skipping generation
2026-08-26 10:11:58,100 - INFO - BSBM dataset already exists in data/bsbm_4, skipping generation
2026-08-26 10:11:58,100 - INFO - BSBM dataset already exists in data/bsbm_5, skipping generation


Running BDSDM generation for size 1...
Running BDSDM generation for size 10...
Running BDSDM generation for size 100...
Running BDSDM generation for size 1000...
Running BDSDM generation for size 10000...
Running BDSDM generation for size 100000...


In [10]:
encoded_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Encoding dataset of size {size}/power {power}...")
    dataset = datasets[power]
    encoded_sizes[power] = dataset.encode(encoding_model)
    #  dataset.get_triple_count(encoded=True)

Encoding dataset of size 1/power 0...
Encoded TTL file already exists at data/bsbm_0/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10/power 1...
Encoded TTL file already exists at data/bsbm_1/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100/power 2...
Encoded TTL file already exists at data/bsbm_2/dataset_encoded.nt, skipping encoding
Encoding dataset of size 1000/power 3...
Encoded TTL file already exists at data/bsbm_3/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10000/power 4...
Encoded TTL file already exists at data/bsbm_4/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100000/power 5...
Encoded TTL file already exists at data/bsbm_5/dataset_encoded.nt, skipping encoding


## BSBM queries

### Simple Use case: 
find 10 products with a specific encoded label


### Complex Use case: 
For a specific product find 10 other similar products via their product label. 


In [11]:
test_label = "house furniture storage container"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
test_tensor.to_literal().n3()

'"{\\"data\\": [0.0019237988162785769, 0.0590696781873703, -0.06259278953075409, -0.008788960985839367, 0.08017757534980774, 0.0004897424369119108, 0.04901154339313507, -0.016156313940882683, -0.045378170907497406, 0.011929638683795929, -0.003551348578184843, -0.003478354075923562, 0.02326440066099167, 0.024490000680088997, -0.023595565930008888, -0.07382390648126602, 0.014226831495761871, -0.00022307882318273187, -0.031117543578147888, 0.07163398712873459, -0.04559384658932686, 0.04442799091339111, 0.00043878634460270405, -0.0040442910976707935, 0.051731135696172714, 0.08436278998851776, -0.0286672692745924, -0.032494235783815384, 0.058949705213308334, -0.005107350647449493, 0.09506019204854965, -0.029170077294111252, -0.07172349095344543, 0.03642681986093521, 0.017889831215143204, 0.07774496078491211, -0.010582796297967434, -0.027199435979127884, -0.014839786104857922, 0.0049964687786996365, -0.051953013986349106, -0.01563340798020363, -0.0004883426008746028, 0.012572056613862514, -0

In [13]:
base_bsbm_set = datasets[3]
db_with_tensor_idx = QleverDBNative(
    id="test",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
    name="QLever",
)
db_no_tensor_idx = QleverDBNative(
    id="test",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
    name="QLever (no Tensor Vocabulary)",
    enable_tensor_index=False,
)
db_fuseki = FusekiDBNative(
    id="test_fuseki",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
    name="Fuseki + RDFTensor",
)
possible_queries = db_with_tensor_idx.get_queries(test_tensor)

from utils.dbs.executable_db import ExecutableDB

dbs: list[ExecutableDB] = [db_with_tensor_idx, db_no_tensor_idx, db_fuseki]
print(possible_queries)


2026-08-26 10:11:58,450 - WARNING - Killing any existing process using port 26043 before starting the server
26043/tcp:          
2026-08-26 10:11:58,513 - INFO - Initialized QLeverDBNative with id=test, port_id=26043, dataset=bsbm, name=QLever, use_encoded_ttl=True, endpoint=http://localhost:26043/test-with-tidx/sparql
2026-08-26 10:11:58,516 - WARNING - Killing any existing process using port 26044 before starting the server
2026-08-26 10:11:58,523 - ERROR - Command failed with return code 1
2026-08-26 10:11:58,524 - INFO - Initialized QLeverDBNative with id=test, port_id=26044, dataset=bsbm, name=QLever (no Tensor Vocabulary), use_encoded_ttl=True, endpoint=http://localhost:26044/test-no-tidx/sparql
2026-08-26 10:11:58,525 - WARNING - Killing any existing process using port 29045 before starting the server
2026-08-26 10:11:58,532 - ERROR - Command failed with return code 1


 2769883{<QUERY_DIFFICULTY.EASY: 'easy'>: {<QUERY_TYPE.EMBEDDED: 'embedded'>: '\nPREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>\nPREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>\nPREFIX dtf: <https://w3id.org/rdf-tensor/functions#>\nPREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>\nPREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>\nSELECT DISTINCT ?product  ?vector ?dist \nWHERE {\n?product rdf:label_embedding ?vector .\n?product bsbmv:productFeature ?feat .\nBIND(dtf:cosineSimilarity(?vector, "{\\"data\\": [0.0019237988162785769, 0.0590696781873703, -0.06259278953075409, -0.008788960985839367, 0.08017757534980774, 0.0004897424369119108, 0.04901154339313507, -0.016156313940882683, -0.045378170907497406, 0.011929638683795929, -0.003551348578184843, -0.003478354075923562, 0.02326440066099167, 0.024490000680088997, -0.023595565930008888, -0.07382390648126602, 0.014226831495761871, -0.00022307882318273187, -0.031117543578147888, 0.0

In [14]:
print(possible_queries[QUERY_DIFFICULTY.EASY][QUERY_TYPE.EMBEDDED])


PREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>
PREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>
SELECT DISTINCT ?product  ?vector ?dist 
WHERE {
?product rdf:label_embedding ?vector .
?product bsbmv:productFeature ?feat .
BIND(dtf:cosineSimilarity(?vector, "{\"data\": [0.0019237988162785769, 0.0590696781873703, -0.06259278953075409, -0.008788960985839367, 0.08017757534980774, 0.0004897424369119108, 0.04901154339313507, -0.016156313940882683, -0.045378170907497406, 0.011929638683795929, -0.003551348578184843, -0.003478354075923562, 0.02326440066099167, 0.024490000680088997, -0.023595565930008888, -0.07382390648126602, 0.014226831495761871, -0.00022307882318273187, -0.031117543578147888, 0.07163398712873459, -0.04559384658932686, 0.04442799091339111, 0.00043878634460270405, -0.004

In [15]:
import os

os.getcwd()

'/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/benchmarks'

In [16]:
import time


with db_fuseki as db:
    timings = []
    for _ in tqdm.tqdm(range(10)):
        start_time = time.time()
        noised_tensor = test_tensor.to_numpy() + np.random.normal(
            scale=0.01, size=test_tensor.to_numpy().shape
        )
        noised_tensor = DataTensor.from_numpy(noised_tensor)

        results = db.query_auto(
            noised_tensor,
            query_difficulty=QUERY_DIFFICULTY.EASY,
            query_type=QUERY_TYPE.EMBEDDED,
        )
        elapsed_time = time.time() - start_time
        timings.append(elapsed_time * 1e9)  # convert to nanoseconds
print("Median elapsed time:", np.median(timings))
results

2026-08-26 10:11:58,680 - WARNING - Killing any existing process using port 29045 before starting the server
2026-08-26 10:11:58,688 - ERROR - Command failed with return code 1
2026-08-26 10:11:58,689 - INFO - Loading dataset into Fuseki server from data/bsbm_3/dataset.nt
2026-08-26 10:11:58,690 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test_fuseki already exists, removing locks if any
2026-08-26 10:11:58,690 - INFO - Stopping server!
2026-08-26 10:11:58,697 - ERROR - Command failed with return code 1
2026-08-26 10:11:58,697 - INFO - Starting Fuseki server port 29045
2026-08-26 10:11:58,698 - ERROR - TENSOR_CP environment variable is not set
2026-08-26 10:11:58,698 - INFO - Using default TENSOR_CP=/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/jena-datatensor/jena-datatensor/target/*
2026-08-26 10:11:58,698 - INFO - Using FUSEKI_HOME=/home/bkantz/apache-jena-fuseki-5.2.0 and TENSOR_CP=/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/jena-datatensor/jena-datatensor/target/*
2026-08-

 2794933Median elapsed time: 615586757.6599121


29045/tcp:          


,product,vector,dist
0,bsbmi:dataFromProducer14/Product668,"{""data"": [0.06415610015392303, 0.0112423654645...",0.3369561705496898
1,bsbmi:dataFromProducer19/Product897,"{""data"": [0.026059485971927643, 0.024894496425...",0.28632440796386094
2,bsbmi:dataFromProducer14/Product665,"{""data"": [-0.00519994692876935, 0.057375498116...",0.27980117790123604
3,bsbmi:dataFromProducer5/Product187,"{""data"": [0.08207502216100693, 0.0040897210128...",0.2504551091270471
4,bsbmi:dataFromProducer14/Product648,"{""data"": [-0.06801607459783554, 0.064525999128...",0.23703017680906277
5,bsbmi:dataFromProducer9/Product439,"{""data"": [-0.11578097194433212, 0.102947957813...",0.23322656647732337
6,bsbmi:dataFromProducer13/Product610,"{""data"": [-0.040756646543741226, 0.08883915096...",0.219612237959987
7,bsbmi:dataFromProducer16/Product776,"{""data"": [-0.05298733338713646, 0.066510133445...",0.21699085544551527
8,bsbmi:dataFromProducer3/Product138,"{""data"": [-0.026387400925159454, 0.05900768190...",0.2143627588834055
9,bsbmi:dataFromProducer14/Product671,"{""data"": [-0.055492501705884933, 0.00027009769...",0.21066843200562144


In [17]:
db_no_tensor_idx.server_log_file

PosixPath('scratch/bsbm/bsbm_3/db/test-no-tidx/test-no-tidx_run.log')

In [18]:
subtimings = db_fuseki.get_timings_per_query()
subtiming_records = []
for i, query_timings in enumerate(subtimings):
    for key, ts in query_timings.items():
        for t in ts:
            subtiming_records.append(
                {"query_id": i - 1, "timing_key": key, "timings": t}
            )
df_subtimings = pd.DataFrame(subtiming_records)
df_timings = pd.DataFrame(
    [
        {"query_id": i, "timing_key": "totalExecution", "timings": t}
        for i, t in enumerate(timings)
    ]
)
full_timings = pd.concat([df_subtimings, df_timings], ignore_index=True)


In [19]:
len(subtimings)

11

In [20]:
full_timings["timing_key"].describe()

count                     206499
unique                         3
top       tensorCosineSimilarity
freq                      205480
Name: timing_key, dtype: object

In [21]:
full_timings.groupby(["query_id", "timing_key"])[
    ["timings"]
].sum().reset_index().sort_values(by=["query_id", "timing_key"])

,query_id,timing_key,timings
0,0,tensorCosineSimilarity,6.789952e+08
1,0,tensorFromString,1.131922e+09
2,0,totalExecution,2.995193e+09
3,1,tensorCosineSimilarity,4.151525e+08
4,1,tensorFromString,2.007355e+06
5,1,totalExecution,5.669479e+08
6,2,tensorCosineSimilarity,4.255405e+08
7,2,tensorFromString,1.025444e+06
8,2,totalExecution,5.379927e+08
9,3,tensorCosineSimilarity,4.844890e+08


In [22]:
repetitions = 128


def run_queries(
    db: ExecutableDB, q_type: QUERY_TYPE, test_tensor: DataTensor, repetitions=128
):
    query_timings_ns = []
    available_qs = db.get_available_query_types()
    if q_type not in available_qs:
        print(f"Query type {q_type} not available for database {db.name}. Skipping.")
        return query_timings_ns
    g = tqdm.tqdm(range(repetitions), total=repetitions, unit="query", leave=False)
    g.set_description(f"Running queries on {db.name}/{q_type}: ")
    for _ in g:
        start_time = time.time()
        noised_tensor = test_tensor.to_numpy() + np.random.normal(
            scale=0.01, size=test_tensor.to_numpy().shape
        )
        noised_tensor = DataTensor.from_numpy(noised_tensor)

        results = db.query_auto(
            noised_tensor,
            query_difficulty=QUERY_DIFFICULTY.EASY,
            query_type=q_type,
        )
        elapsed_time = time.time() - start_time
        query_timings_ns.append(elapsed_time * 1e9)
    return query_timings_ns


all_timings = None
for q_type in [QUERY_TYPE.EMBEDDED, QUERY_TYPE.INDEX]:
    for db in dbs:
        db.clear_log()
        with db as db_instance:
            query_timings_ns = run_queries(
                db_instance, q_type, test_tensor, repetitions=1
            )
            if len(query_timings_ns) == 0:
                continue
            subtimings = db_instance.get_timings_per_query()
            subtiming_records = []
            for i, qt in enumerate(subtimings):
                for key, ts in qt.items():
                    for t in ts:
                        subtiming_records.append(
                            {
                                "query_id": i - 1,
                                "timing_key": key,
                                "timings": t,
                                "engine": db_instance.name,
                                "query_type": q_type.name.lower(),
                            }
                        )
            df_subtimings = pd.DataFrame(subtiming_records)
            df_timings = pd.DataFrame(
                [
                    {
                        "query_id": i,
                        "timing_key": "totalExecution",
                        "timings": t,
                        "engine": db_instance.name,
                        "query_type": q_type.name.lower(),
                    }
                    for i, t in enumerate(query_timings_ns)
                ]
            )
            full_timings = pd.concat([df_subtimings, df_timings], ignore_index=True)

            full_timings_sums = (
                full_timings.groupby(
                    ["query_id", "timing_key", "engine", "query_type"]
                )[["timings"]]
                .agg(["sum", "count"])
                .reset_index()
                .sort_values(by=["query_id", "timing_key"])
            )
            all_timings = (
                pd.concat([all_timings, full_timings_sums], ignore_index=True)
                if all_timings is not None
                else full_timings_sums
            )

2026-08-26 10:12:09,395 - INFO - Setting up QLeverDBNative db_dir=scratch/bsbm/bsbm_3/db/test-with-tidx, base_dir=scratch/bsbm/bsbm_3
2026-08-26 10:12:09,396 - INFO - Logging QLever setup to scratch/bsbm/bsbm_3/db/test-with-tidx/test-with-tidx_run.log
2026-08-26 10:12:09,396 - INFO - Loading dataset into QLever server from data/bsbm_3/dataset.nt
2026-08-26 10:12:09,401 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test-with-tidx already exists!
2026-08-26 10:12:09,402 - INFO - Stopping server!
2026-08-26 10:12:09,409 - ERROR - Command failed with return code 1
2026-08-26 10:12:09,410 - INFO - Starting QLever server on port 26043
2026-08-26 10:12:09,410 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-08-26 10:12:09,412 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/test-with-tidx/sparql)
2026-08-26 10:12:10,427 - INFO - Server is up and 

 2795107

2026-08-26 10:12:12,032 - INFO - Server is up and responding to queries
2026-08-26 10:12:12,795 - INFO - Stopping server!                                                                       
26044/tcp:          
2026-08-26 10:12:12,871 - WARNING - Killing any existing process using port 29045 before starting the server
2026-08-26 10:12:12,947 - ERROR - Command failed with return code 1
2026-08-26 10:12:12,948 - INFO - Loading dataset into Fuseki server from data/bsbm_3/dataset.nt
2026-08-26 10:12:12,948 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test_fuseki already exists, removing locks if any
2026-08-26 10:12:12,949 - INFO - Stopping server!
2026-08-26 10:12:13,020 - ERROR - Command failed with return code 1
2026-08-26 10:12:13,021 - INFO - Starting Fuseki server port 29045
2026-08-26 10:12:13,021 - ERROR - TENSOR_CP environment variable is not set
2026-08-26 10:12:13,022 - INFO - Using default TENSOR_CP=/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/jena-datatensor/jena-datat

 2795140

2026-08-26 10:12:14,109 - INFO - Server is up and responding to queries
2026-08-26 10:12:16,478 - INFO - Stopping server!                                                            
29045/tcp:          
2026-08-26 10:12:16,556 - INFO - Setting up QLeverDBNative db_dir=scratch/bsbm/bsbm_3/db/test-with-tidx, base_dir=scratch/bsbm/bsbm_3
2026-08-26 10:12:16,557 - INFO - Logging QLever setup to scratch/bsbm/bsbm_3/db/test-with-tidx/test-with-tidx_run.log
2026-08-26 10:12:16,557 - INFO - Loading dataset into QLever server from data/bsbm_3/dataset.nt
2026-08-26 10:12:16,558 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test-with-tidx already exists!
2026-08-26 10:12:16,561 - INFO - Stopping server!
2026-08-26 10:12:16,640 - ERROR - Command failed with return code 1
2026-08-26 10:12:16,640 - INFO - Starting QLever server on port 26043
2026-08-26 10:12:16,641 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-08-26 10:12:

...

2026-08-26 10:12:18,924 - INFO - Server is up and responding to queries
2026-08-26 10:12:19,025 - INFO - Stopping server!                                                            
26044/tcp:          
2026-08-26 10:12:19,108 - WARNING - Killing any existing process using port 29045 before starting the server
2026-08-26 10:12:19,178 - ERROR - Command failed with return code 1
2026-08-26 10:12:19,179 - INFO - Loading dataset into Fuseki server from data/bsbm_3/dataset.nt
2026-08-26 10:12:19,180 - WARNING - DB directory scratch/bsbm/bsbm_3/db/test_fuseki already exists, removing locks if any
2026-08-26 10:12:19,180 - INFO - Stopping server!
2026-08-26 10:12:19,245 - ERROR - Command failed with return code 1
2026-08-26 10:12:19,246 - INFO - Starting Fuseki server port 29045
2026-08-26 10:12:19,246 - ERROR - TENSOR_CP environment variable is not set
2026-08-26 10:12:19,247 - INFO - Using default TENSOR_CP=/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB/jena-datatensor/jena-datatensor/targe

 2795377

2026-08-26 10:12:20,795 - INFO - Server is up and responding to queries
2026-08-26 10:12:20,796 - INFO - Stopping server!


Query type QUERY_TYPE.INDEX not available for database Fuseki + RDFTensor. Skipping.
 2795427

29045/tcp:          


In [23]:
from utils.format import map_df_readable

all_timings.to_csv("scratch/sub_bsbm_timings.csv", index=False)
all_timings_mapped = map_df_readable(all_timings).reset_index()
all_timings_mapped

index query_id                 timing_key            engine query_type  \
                                                                            
0      0       -1  processQueryAndSendResult    \systemname-TV   Embedded   
1      1        0  processQueryAndSendResult    \systemname-TV   Embedded   
2      2        0             readTensorData    \systemname-TV   Embedded   
3      3        0           readWordFromDisk    \systemname-TV   Embedded   
4      4        0     tensorCosineSimilarity    \systemname-TV   Embedded   
5      5        0           tensorFromBuffer    \systemname-TV   Embedded   
6      6        0           tensorFromString    \systemname-TV   Embedded   
7      7        0             totalExecution    \systemname-TV   Embedded   
8      8       -1  processQueryAndSendResult  \systemname-Base   Embedded   
9      9        0  processQueryAndSendResult  \systemname-Base   Embedded   
10    10        0           readWordFromDisk  \systemname-Base   Embedded   
11    11        0     tensorCosineSimilarity  \systemname-Base   Embedded   
12    12        0           tensorFromString  \systemname-Base   Embedded   
13    13        0             totalExecution  \systemname-Base   Embedded   
14    14        0     tensorCosineSimilarity         RDFTensor   Embedded   
15    15        0           tensorFromString         RDFTensor   Embedded   
16    16        0             totalExecution         RDFTensor   Embedded   
17    17       -1  processQueryAndSendResult    \systemname-TV      Index   
18    18        0  processQueryAndSendResult    \systemname-TV      Index   
19    19        0             readTensorData    \systemname-TV      Index   
20    20        0           readWordFromDisk    \systemname-TV      Index   
21    21        0           tensorFromBuffer    \systemname-TV      Index   
22    22        0           tensorFromString    \systemname-TV      Index   
23    23        0   tensorIndexComputeResult    \systemname-TV      Index   
24    24        0             totalExecution    \systemname-TV      Index   
25    25       -1  processQueryAndSendResult  \systemname-Base      Index   
26    26        0  processQueryAndSendResult  \systemname-Base      Index   
27    27        0           readWordFromDisk  \systemname-Base      Index   
28    28        0           tensorFromString  \systemname-Base      Index   
29    29        0   tensorIndexComputeResult  \systemname-Base      Index   
30    30        0             totalExecution  \systemname-Base      Index   

         timings         
             sum  count  
0   7.456323e+06      1  
1   3.227511e+08      1  
2   2.240732e+08  20548  
3   7.928277e+06    115  
4   3.187112e+06  20548  
5   1.047442e+06  20548  
6   1.717400e+04      3  
7   3.466618e+08      1  
8   4.449794e+06      1  
9   5.991239e+08      1  
10  2.481776e+08  20663  
11  2.743076e+06  20548  
12  6.834801e+06  20551  
13  6.177485e+08      1  
14  6.447741e+08  20548  
15  1.087755e+09   1000  
16  2.300757e+09      1  
17  1.842349e+06      1  
18  3.916281e+07      1  
19  2.846778e+06   1000  
20  3.696745e+06    136  
21  4.209000e+04   1000  
22  8.055000e+03      1  
23  3.583951e+07      1  
24  5.382466e+07      1  
25  2.271501e+06      1  
26  5.889653e+07      1  
27  5.285489e+06   1136  
28  3.994080e+05   1001  
29  5.699114e+07      1  
30  7.745767e+07      1

In [24]:
all_timings_mapped["timing_key"]

0     processQueryAndSendResult
1     processQueryAndSendResult
2                readTensorData
3              readWordFromDisk
4        tensorCosineSimilarity
5              tensorFromBuffer
6              tensorFromString
7                totalExecution
8     processQueryAndSendResult
9     processQueryAndSendResult
10             readWordFromDisk
11       tensorCosineSimilarity
12             tensorFromString
13               totalExecution
14       tensorCosineSimilarity
15             tensorFromString
16               totalExecution
17    processQueryAndSendResult
18    processQueryAndSendResult
19               readTensorData
20             readWordFromDisk
21             tensorFromBuffer
22             tensorFromString
23     tensorIndexComputeResult
24               totalExecution
25    processQueryAndSendResult
26    processQueryAndSendResult
27             readWordFromDisk
28             tensorFromString
29     tensorIndexComputeResult
30               totalExecution
Name: ti

In [25]:
all_timings_mapped["engine"].unique()

<ArrowStringArray>
['\systemname-TV', '\systemname-Base', 'RDFTensor']
Length: 3, dtype: str

In [26]:
all_timings_mapped[
    (all_timings_mapped["engine"] == "RDFTensor") & (all_timings_mapped["query_type"] == "Embedded")
]

index query_id              timing_key     engine query_type       timings  \
                                                                          sum   
14    14        0  tensorCosineSimilarity  RDFTensor   Embedded  6.447741e+08   
15    15        0        tensorFromString  RDFTensor   Embedded  1.087755e+09   
16    16        0          totalExecution  RDFTensor   Embedded  2.300757e+09   

           
    count  
14  20548  
15   1000  
16      1

In [27]:
all_timings_mapped[
    (all_timings_mapped["engine"] == "\systemname-Base")
    & (all_timings_mapped["query_type"] == "Embedded")
]

<>:2: SyntaxWarning: invalid escape sequence '\s'
<>:2: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2794777/3222887891.py:2: SyntaxWarning: invalid escape sequence '\s'
  (all_timings_mapped["engine"] == "\systemname-Base")


index query_id                 timing_key            engine query_type  \
                                                                            
8      8       -1  processQueryAndSendResult  \systemname-Base   Embedded   
9      9        0  processQueryAndSendResult  \systemname-Base   Embedded   
10    10        0           readWordFromDisk  \systemname-Base   Embedded   
11    11        0     tensorCosineSimilarity  \systemname-Base   Embedded   
12    12        0           tensorFromString  \systemname-Base   Embedded   
13    13        0             totalExecution  \systemname-Base   Embedded   

         timings         
             sum  count  
8   4.449794e+06      1  
9   5.991239e+08      1  
10  2.481776e+08  20663  
11  2.743076e+06  20548  
12  6.834801e+06  20551  
13  6.177485e+08      1

In [28]:
all_timings_mapped


index query_id                 timing_key            engine query_type  \
                                                                            
0      0       -1  processQueryAndSendResult    \systemname-TV   Embedded   
1      1        0  processQueryAndSendResult    \systemname-TV   Embedded   
2      2        0             readTensorData    \systemname-TV   Embedded   
3      3        0           readWordFromDisk    \systemname-TV   Embedded   
4      4        0     tensorCosineSimilarity    \systemname-TV   Embedded   
5      5        0           tensorFromBuffer    \systemname-TV   Embedded   
6      6        0           tensorFromString    \systemname-TV   Embedded   
7      7        0             totalExecution    \systemname-TV   Embedded   
8      8       -1  processQueryAndSendResult  \systemname-Base   Embedded   
9      9        0  processQueryAndSendResult  \systemname-Base   Embedded   
10    10        0           readWordFromDisk  \systemname-Base   Embedded   
11    11        0     tensorCosineSimilarity  \systemname-Base   Embedded   
12    12        0           tensorFromString  \systemname-Base   Embedded   
13    13        0             totalExecution  \systemname-Base   Embedded   
14    14        0     tensorCosineSimilarity         RDFTensor   Embedded   
15    15        0           tensorFromString         RDFTensor   Embedded   
16    16        0             totalExecution         RDFTensor   Embedded   
17    17       -1  processQueryAndSendResult    \systemname-TV      Index   
18    18        0  processQueryAndSendResult    \systemname-TV      Index   
19    19        0             readTensorData    \systemname-TV      Index   
20    20        0           readWordFromDisk    \systemname-TV      Index   
21    21        0           tensorFromBuffer    \systemname-TV      Index   
22    22        0           tensorFromString    \systemname-TV      Index   
23    23        0   tensorIndexComputeResult    \systemname-TV      Index   
24    24        0             totalExecution    \systemname-TV      Index   
25    25       -1  processQueryAndSendResult  \systemname-Base      Index   
26    26        0  processQueryAndSendResult  \systemname-Base      Index   
27    27        0           readWordFromDisk  \systemname-Base      Index   
28    28        0           tensorFromString  \systemname-Base      Index   
29    29        0   tensorIndexComputeResult  \systemname-Base      Index   
30    30        0             totalExecution  \systemname-Base      Index   

         timings         
             sum  count  
0   7.456323e+06      1  
1   3.227511e+08      1  
2   2.240732e+08  20548  
3   7.928277e+06    115  
4   3.187112e+06  20548  
5   1.047442e+06  20548  
6   1.717400e+04      3  
7   3.466618e+08      1  
8   4.449794e+06      1  
9   5.991239e+08      1  
10  2.481776e+08  20663  
11  2.743076e+06  20548  
12  6.834801e+06  20551  
13  6.177485e+08      1  
14  6.447741e+08  20548  
15  1.087755e+09   1000  
16  2.300757e+09      1  
17  1.842349e+06      1  
18  3.916281e+07      1  
19  2.846778e+06   1000  
20  3.696745e+06    136  
21  4.209000e+04   1000  
22  8.055000e+03      1  
23  3.583951e+07      1  
24  5.382466e+07      1  
25  2.271501e+06      1  
26  5.889653e+07      1  
27  5.285489e+06   1136  
28  3.994080e+05   1001  
29  5.699114e+07      1  
30  7.745767e+07      1

In [29]:
def map_group_stats(df: pd.DataFrame, value_col: str = "timings"):
    stat_mapping = {
        "totalExecution": "Total (ms / ct.)",
        "readTensorData": "Read Tensors (ms / ct.)",
        "readWordFromDisk": "Read RDF (ms / ct.)",
        "tensorCosineSimilarity": "Cosine Similarity (ms / ct.)",
        "tensorFromBuffer": "Parse from Buffer (ms / ct.)",
        "tensorFromString": "Parse from String (ms / ct.)",
        "tensorIndexComputeResult": "Index Lookup (ms / ct.)",
    }
    df_cp = df.copy()
    df_cp["timing_key"] = df_cp["timing_key"].map(stat_mapping)
    
    df_cp = df_cp[df_cp["timing_key"].notna()]

    # timing mappings to columns

    df_grouped = (
        df_cp.groupby(["engine", "query_type", "timing_key"])[[("timings", "sum"), ("timings", "count")]]
        .median()
        .reset_index()
        .sort_values(by=["engine", "query_type", "timing_key"])
    )
    df_grouped[("timings", "sum")] = (
        df_grouped[("timings", "sum")] / 1e6
    )  # convert to ms
    return df_grouped


all_timings_grouped = map_group_stats(all_timings_mapped)
all_timings_grouped


engine query_type                    timing_key      timings  \
                                                                       sum   
0          RDFTensor   Embedded  Cosine Similarity (ms / ct.)   644.774065   
1          RDFTensor   Embedded  Parse from String (ms / ct.)  1087.754546   
2          RDFTensor   Embedded              Total (ms / ct.)  2300.756931   
3   \systemname-Base   Embedded  Cosine Similarity (ms / ct.)     2.743076   
4   \systemname-Base   Embedded  Parse from String (ms / ct.)     6.834801   
5   \systemname-Base   Embedded           Read RDF (ms / ct.)   248.177581   
6   \systemname-Base   Embedded              Total (ms / ct.)   617.748499   
7   \systemname-Base      Index       Index Lookup (ms / ct.)    56.991138   
8   \systemname-Base      Index  Parse from String (ms / ct.)     0.399408   
9   \systemname-Base      Index           Read RDF (ms / ct.)     5.285489   
10  \systemname-Base      Index              Total (ms / ct.)    77.457666   
11    \systemname-TV   Embedded  Cosine Similarity (ms / ct.)     3.187112   
12    \systemname-TV   Embedded  Parse from Buffer (ms / ct.)     1.047442   
13    \systemname-TV   Embedded  Parse from String (ms / ct.)     0.017174   
14    \systemname-TV   Embedded           Read RDF (ms / ct.)     7.928277   
15    \systemname-TV   Embedded       Read Tensors (ms / ct.)   224.073240   
16    \systemname-TV   Embedded              Total (ms / ct.)   346.661806   
17    \systemname-TV      Index       Index Lookup (ms / ct.)    35.839512   
18    \systemname-TV      Index  Parse from Buffer (ms / ct.)     0.042090   
19    \systemname-TV      Index  Parse from String (ms / ct.)     0.008055   
20    \systemname-TV      Index           Read RDF (ms / ct.)     3.696745   
21    \systemname-TV      Index       Read Tensors (ms / ct.)     2.846778   
22    \systemname-TV      Index              Total (ms / ct.)    53.824663   

             
      count  
0   20548.0  
1    1000.0  
2       1.0  
3   20548.0  
4   20551.0  
5   20663.0  
6       1.0  
7       1.0  
8    1001.0  
9    1136.0  
10      1.0  
11  20548.0  
12  20548.0  
13      3.0  
14    115.0  
15  20548.0  
16      1.0  
17      1.0  
18   1000.0  
19      1.0  
20    136.0  
21   1000.0  
22      1.0

In [30]:
def multileve_timings_to_str(df: pd.DataFrame, col="timings"):
    df_cp = df.copy()

    def combine_multi_level(row):
        # print(row)
        summed =f"{row['sum']:.2f}" if not pd.isna(row["sum"]) else "-"
        counts = int(row["count"]) if not pd.isna(row["count"]) else "-"
        return f"{summed} ({counts})"

    df_cp["combined"] = df_cp[col].apply(combine_multi_level, axis=1)
    return df_cp


all_timings_grouped_combined = multileve_timings_to_str(all_timings_grouped).drop(columns=[("timings", "sum"), ("timings", "count")]).droplevel(1, axis=1)
all_timings_grouped_combined

,engine,query_type,timing_key,combined
0,RDFTensor,Embedded,Cosine Similarity (ms / ct.),644.77 (20548)
1,RDFTensor,Embedded,Parse from String (ms / ct.),1087.75 (1000)
2,RDFTensor,Embedded,Total (ms / ct.),2300.76 (1)
3,\systemname-Base,Embedded,Cosine Similarity (ms / ct.),2.74 (20548)
4,\systemname-Base,Embedded,Parse from String (ms / ct.),6.83 (20551)
5,\systemname-Base,Embedded,Read RDF (ms / ct.),248.18 (20663)
6,\systemname-Base,Embedded,Total (ms / ct.),617.75 (1)
7,\systemname-Base,Index,Index Lookup (ms / ct.),56.99 (1)
8,\systemname-Base,Index,Parse from String (ms / ct.),0.40 (1001)
9,\systemname-Base,Index,Read RDF (ms / ct.),5.29 (1136)


In [31]:
all_timings_pivot = (
    all_timings_grouped_combined.pivot_table(
        index=["engine", "query_type"], columns=["timing_key"], values="combined", aggfunc=lambda x: " | ".join(x)
    )
    .reset_index()
    .fillna("-")
    .set_index(["engine", "query_type"])
)
all_timings_pivot

timing_key                  Cosine Similarity (ms / ct.)  \
engine           query_type                                
RDFTensor        Embedded                 644.77 (20548)   
\systemname-Base Embedded                   2.74 (20548)   
                 Index                                 -   
\systemname-TV   Embedded                   3.19 (20548)   
                 Index                                 -   

timing_key                  Index Lookup (ms / ct.)  \
engine           query_type                           
RDFTensor        Embedded                         -   
\systemname-Base Embedded                         -   
                 Index                    56.99 (1)   
\systemname-TV   Embedded                         -   
                 Index                    35.84 (1)   

timing_key                  Parse from Buffer (ms / ct.)  \
engine           query_type                                
RDFTensor        Embedded                              -   
\systemname-Base Embedded                              -   
                 Index                                 -   
\systemname-TV   Embedded                   1.05 (20548)   
                 Index                       0.04 (1000)   

timing_key                  Parse from String (ms / ct.) Read RDF (ms / ct.)  \
engine           query_type                                                    
RDFTensor        Embedded                 1087.75 (1000)                   -   
\systemname-Base Embedded                   6.83 (20551)      248.18 (20663)   
                 Index                       0.40 (1001)         5.29 (1136)   
\systemname-TV   Embedded                       0.02 (3)          7.93 (115)   
                 Index                          0.01 (1)          3.70 (136)   

timing_key                  Read Tensors (ms / ct.) Total (ms / ct.)  
engine           query_type                                           
RDFTensor        Embedded                         -      2300.76 (1)  
\systemname-Base Embedded                         -       617.75 (1)  
                 Index                            -        77.46 (1)  
\systemname-TV   Embedded            224.07 (20548)       346.66 (1)  
                 Index                  2.85 (1000)        53.82 (1)

In [32]:
all_timings_pivot.columns.name = None
all_timings_pivot.index.rename(["Engine", "Query Type"], inplace=True)
all_timings_pivot

Cosine Similarity (ms / ct.)  \
Engine           Query Type                                
RDFTensor        Embedded                 644.77 (20548)   
\systemname-Base Embedded                   2.74 (20548)   
                 Index                                 -   
\systemname-TV   Embedded                   3.19 (20548)   
                 Index                                 -   

                            Index Lookup (ms / ct.)  \
Engine           Query Type                           
RDFTensor        Embedded                         -   
\systemname-Base Embedded                         -   
                 Index                    56.99 (1)   
\systemname-TV   Embedded                         -   
                 Index                    35.84 (1)   

                            Parse from Buffer (ms / ct.)  \
Engine           Query Type                                
RDFTensor        Embedded                              -   
\systemname-Base Embedded                              -   
                 Index                                 -   
\systemname-TV   Embedded                   1.05 (20548)   
                 Index                       0.04 (1000)   

                            Parse from String (ms / ct.) Read RDF (ms / ct.)  \
Engine           Query Type                                                    
RDFTensor        Embedded                 1087.75 (1000)                   -   
\systemname-Base Embedded                   6.83 (20551)      248.18 (20663)   
                 Index                       0.40 (1001)         5.29 (1136)   
\systemname-TV   Embedded                       0.02 (3)          7.93 (115)   
                 Index                          0.01 (1)          3.70 (136)   

                            Read Tensors (ms / ct.) Total (ms / ct.)  
Engine           Query Type                                           
RDFTensor        Embedded                         -      2300.76 (1)  
\systemname-Base Embedded                         -       617.75 (1)  
                 Index                            -        77.46 (1)  
\systemname-TV   Embedded            224.07 (20548)       346.66 (1)  
                 Index                  2.85 (1000)        53.82 (1)

In [33]:
with open(figures_dir / "sub_bsbm_timings.tex", "w") as f:
    f.write(
        all_timings_pivot[[c for c in all_timings_pivot.columns if "Read" not in c]].style.to_latex(
            caption="Median timings (ms) and counts for each query type and engine.",
            label="tab:subtimings",
            position="htbp",
            hrules=True,
            column_format="ll" + "r" * len(all_timings_pivot.columns),
        )
    )

In [34]:
counts = []
for power, size in zip(powers, sizes):
    difficulties = [
        QUERY_DIFFICULTY.HARD,
        QUERY_DIFFICULTY.EASY,
    ]
    dataset = datasets[power]
    db = QleverDBNative(
        id="timing-qlever",
        base_dir=Path(f"./scratch/bsbm/{dataset.base_dir.name}"),
        dataset=dataset,
        use_encoded_ttl=True,
    )
    with db:
        full_number_of_tensors = db.query("""
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
SELECT (COUNT(?v) AS ?count) WHERE {
    {
    SELECT ?s ?v WHERE {          
    ?s rdf:label_embedding ?v .
    }
    } UNION {
            SELECT ?s ?v WHERE {
        ?s rdf:comment_embedding ?v .}
    }
}
""")["count"].values[0]
        full_size = db.get_triple_count()
        counts.append(
            {
                "power": power,
                "size": size,
                "full_size": full_size,
                "full_number_of_tensors": full_number_of_tensors,
            }
        )
counts_df = pd.DataFrame(counts)
counts_df

2026-08-26 10:12:21,633 - WARNING - Killing any existing process using port 26046 before starting the server
2026-08-26 10:12:21,641 - ERROR - Command failed with return code 1
2026-08-26 10:12:21,642 - INFO - Initialized QLeverDBNative with id=timing-qlever, port_id=26046, dataset=bsbm, name=QLever Native (Extended), use_encoded_ttl=True, endpoint=http://localhost:26046/timing-qlever-with-tidx/sparql
2026-08-26 10:12:21,642 - INFO - Setting up QLeverDBNative db_dir=scratch/bsbm/bsbm_0/db/timing-qlever-with-tidx, base_dir=scratch/bsbm/bsbm_0
2026-08-26 10:12:21,642 - INFO - Logging QLever setup to scratch/bsbm/bsbm_0/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-08-26 10:12:21,642 - INFO - Loading dataset into QLever server from data/bsbm_0/dataset.nt
2026-08-26 10:12:21,643 - WARNING - DB directory scratch/bsbm/bsbm_0/db/timing-qlever-with-tidx already exists!
2026-08-26 10:12:21,643 - INFO - Stopping server!


2026-08-26 10:12:21,650 - ERROR - Command failed with return code 1
2026-08-26 10:12:21,651 - INFO - Starting QLever server on port 26046
2026-08-26 10:12:21,651 - INFO - Running command: qlever-server -i timing-qlever-with-tidx --port 26046 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-08-26 10:12:21,655 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26046/timing-qlever-with-tidx/sparql)
2026-08-26 10:12:22,663 - INFO - Server is up and responding to queries
2026-08-26 10:12:22,673 - INFO - Stopping server!
26046/tcp:          
2026-08-26 10:12:22,736 - WARNING - Killing any existing process using port 26047 before starting the server
2026-08-26 10:12:22,742 - ERROR - Command failed with return code 1
2026-08-26 10:12:22,742 - INFO - Initialized QLeverDBNative with id=timing-qlever, port_id=26047, dataset=bsbm, name=QLever Native (Extended), use_encoded_ttl=True, endpoint=http://localhost:26047/timing-qlever-wi

 2795557

2026-08-26 10:12:23,754 - INFO - Server is up and responding to queries
2026-08-26 10:12:23,762 - INFO - Stopping server!
26047/tcp:          
2026-08-26 10:12:23,839 - WARNING - Killing any existing process using port 26048 before starting the server
2026-08-26 10:12:23,845 - ERROR - Command failed with return code 1
2026-08-26 10:12:23,845 - INFO - Initialized QLeverDBNative with id=timing-qlever, port_id=26048, dataset=bsbm, name=QLever Native (Extended), use_encoded_ttl=True, endpoint=http://localhost:26048/timing-qlever-with-tidx/sparql
2026-08-26 10:12:23,846 - INFO - Setting up QLeverDBNative db_dir=scratch/bsbm/bsbm_2/db/timing-qlever-with-tidx, base_dir=scratch/bsbm/bsbm_2
2026-08-26 10:12:23,846 - INFO - Logging QLever setup to scratch/bsbm/bsbm_2/db/timing-qlever-with-tidx/timing-qlever-with-tidx_run.log
2026-08-26 10:12:23,846 - INFO - Loading dataset into QLever server from data/bsbm_2/dataset.nt
2026-08-26 10:12:23,846 - WARNING - DB directory scratch/bsbm/bsbm_2/db/timin

 2795599

...

 2795872

2026-08-26 10:12:28,230 - INFO - Server is up and responding to queries
2026-08-26 10:12:28,306 - INFO - Stopping server!


 2795921

26051/tcp:          


,power,size,full_size,full_number_of_tensors
0,0,1,2418,576
1,1,10,5595,608
2,2,100,42417,2240
3,3,1000,383751,11840
4,4,10000,3577077,42304
5,5,100000,35177974,305792


In [35]:
counts_df = counts_df.astype(int)
counts_df

,power,size,full_size,full_number_of_tensors
0,0,1,2418,576
1,1,10,5595,608
2,2,100,42417,2240
3,3,1000,383751,11840
4,4,10000,3577077,42304
5,5,100000,35177974,305792


In [36]:
out_dir = Path("./scratch/results")
out_dir.mkdir(parents=True, exist_ok=True)
from utils.helpers import pretty_print_counts

pretty_print_counts(counts_df, out_dir / "bsbm_counts.tex")

['power', 'size', 'full_size', 'full_number_of_tensors'] []


,Power,Generation $t$,$n$,$n_{tensors}$
0,0,1,2418,576
1,1,10,5595,608
2,2,100,42417,2240
3,3,1000,383751,11840
4,4,10000,3577077,42304
5,5,100000,35177974,305792
